<a href="https://colab.research.google.com/github/iipingebnaa/Intensive-Big-Data-Analytics-Project/blob/main/02_Architecture%20%26%20Storage/Architecture_%26_Storage_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Role: Data Engineer

Bernadette Iipinge
226162389

Step 1: Environment Setup

In [ ]:
# Install the required libraries
!pip install -q pyspark==3.5.1 findspark

# Import required libraries
import os
import findspark
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("PySpark version:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
PySpark version: 3.5.1
PySpark location: /usr/local/lib/python3.13/dist-packages/pyspark/__init__.py


Step 2: Create Spark Session

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("RetailMart_Architecture_Storage") \
    .config("spark.sql.warehouse.dir", "/content/spark-warehouse") \
    .getOrCreate()

print("Spark session created successfully!")
print("Spark version:", spark.version)

Spark session created successfully!
Spark version: 3.5.1


Step 3: Load Cleaned Datasets

In [5]:
from google.colab import files

uploaded = files.upload()

Saving customer_feedback_cleaned.csv to customer_feedback_cleaned.csv
Saving retailmart_transactions_cleaned.csv to retailmart_transactions_cleaned.csv


Step 3.1: Define Dataset Paths

In [6]:
# Define paths to the cleaned datasets
transactions_path = "/content/retailmart_transactions_cleaned.csv"
feedback_path = "/content/customer_feedback_cleaned.csv"

print("Transaction file:", transactions_path)
print("Feedback file:", feedback_path)

Transaction file: /content/retailmart_transactions_cleaned.csv
Feedback file: /content/customer_feedback_cleaned.csv


Step 3.2: Load Transaction Data

In [7]:
# Load the cleaned transaction dataset
df_transactions = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(transactions_path)

print("Transaction dataset loaded successfully!")

Transaction dataset loaded successfully!


Step 3.3: Load Customer Feedback Data

In [8]:
# Load the cleaned customer feedback dataset
df_feedback = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(feedback_path)

print("Customer feedback dataset loaded successfully!")

Customer feedback dataset loaded successfully!


Step 3.4: Verify Loaded Datasets

In [9]:
# Count rows and columns in each dataset
print("Transaction dataset:")
print("Rows:", df_transactions.count())
print("Columns:", len(df_transactions.columns))

print("\nCustomer feedback dataset:")
print("Rows:", df_feedback.count())
print("Columns:", len(df_feedback.columns))

Transaction dataset:
Rows: 973266
Columns: 15

Customer feedback dataset:
Rows: 250000
Columns: 14


Step 4: Inspect Dataset Structure

In [10]:
# Display the transaction dataset schema
print("Transaction Dataset Schema:")
df_transactions.printSchema()

# Display the customer feedback dataset schema
print("\nCustomer Feedback Dataset Schema:")
df_feedback.printSchema()

Transaction Dataset Schema:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- store_location: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- loyalty_years: integer (nullable = true)
 |-- num_visits: integer (nullable = true)
 |-- churn: string (nullable = true)


Customer Feedback Dataset Schema:
root
 |-- feedback_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- platform: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)


Step 5: Create RetailMart Database

In [11]:
# Create the RetailMart database
spark.sql("CREATE DATABASE IF NOT EXISTS retailmart_db")

# Set the database as the current database
spark.sql("USE retailmart_db")

# Verify the current database
print("Current database:", spark.catalog.currentDatabase())

Current database: retailmart_db


Step 6: Create Managed Tables

In [12]:
# Save transaction data as a managed table
df_transactions.write \
    .mode("overwrite") \
    .saveAsTable("retailmart_db.transactions_table")

print("Transactions table created successfully!")

# Save customer feedback data as a managed table
df_feedback.write \
    .mode("overwrite") \
    .saveAsTable("retailmart_db.feedback_table")

print("Customer feedback table created successfully!")

Transactions table created successfully!
Customer feedback table created successfully!


Step 7: Verify Stored Tables

In [13]:
# List tables in the RetailMart database
spark.sql("SHOW TABLES IN retailmart_db").show(truncate=False)

+-------------+------------------+-----------+
|namespace    |tableName         |isTemporary|
+-------------+------------------+-----------+
|retailmart_db|feedback_table    |false      |
|retailmart_db|transactions_table|false      |
+-------------+------------------+-----------+



Step 8: Query Stored Tables

In [15]:
# Display sample transaction records
print("Sample transaction records:")
spark.sql("""
    SELECT transaction_id, transaction_date, product_category, total_amount
    FROM retailmart_db.transactions_table
    LIMIT 5
""").show(truncate=False)

# Display sample customer feedback records
print("Sample customer feedback records:")
spark.sql("""
    SELECT feedback_id, feedback_rating, feedback_sentiment, platform
    FROM retailmart_db.feedback_table
    LIMIT 5
""").show(truncate=False)

Sample transaction records:
+--------------+-------------------+----------------+------------+
|transaction_id|transaction_date   |product_category|total_amount|
+--------------+-------------------+----------------+------------+
|TXN-00000001  |2022-04-03 10:47:06|Clothing        |43.13       |
|TXN-00000002  |2021-03-03 09:13:14|Clothing        |55.36       |
|TXN-00000003  |2022-03-28 15:37:17|Groceries       |30.84       |
|TXN-00000004  |2022-07-24 10:13:48|Home & Kitchen  |87.83       |
|TXN-00000005  |2024-05-21 12:51:02|Toys            |34.94       |
+--------------+-------------------+----------------+------------+

Sample customer feedback records:
+------------------------------------+---------------+------------------+----------+
|feedback_id                         |feedback_rating|feedback_sentiment|platform  |
+------------------------------------+---------------+------------------+----------+
|6acd1783-8170-4a6e-8a7f-d942ffb86cf3|3              |0.129             |Web   

Step 9: Prepare Data for Partitioning

In [16]:
# Extract year and month from the transaction date
df_transactions_partitioned = df_transactions \
    .withColumn("transaction_year", F.year("transaction_date")) \
    .withColumn("transaction_month", F.month("transaction_date"))

# Display the new columns
df_transactions_partitioned.select(
    "transaction_date",
    "transaction_year",
    "transaction_month"
).show(5, truncate=False)

+-------------------+----------------+-----------------+
|transaction_date   |transaction_year|transaction_month|
+-------------------+----------------+-----------------+
|2022-04-03 10:47:06|2022            |4                |
|2021-03-03 09:13:14|2021            |3                |
|2022-03-28 15:37:17|2022            |3                |
|2022-07-24 10:13:48|2022            |7                |
|2024-05-21 12:51:02|2024            |5                |
+-------------------+----------------+-----------------+
only showing top 5 rows



Step 10: Save Partitioned Data

In [17]:
# Save transactions partitioned by year and month
df_transactions_partitioned.write \
    .mode("overwrite") \
    .partitionBy("transaction_year", "transaction_month") \
    .format("parquet") \
    .saveAsTable("retailmart_db.transactions_partitioned")

print("Partitioned transaction table created successfully!")

Partitioned transaction table created successfully!


Step 11: Verify Partitioned Storage

In [18]:
# Verify the partitioned table exists
spark.sql("""
    SHOW TABLES IN retailmart_db
""").show(truncate=False)

# Inspect the partitioned table schema
print("Partitioned transaction table schema:")
spark.table("retailmart_db.transactions_partitioned").printSchema()

# Display sample records
spark.table("retailmart_db.transactions_partitioned") \
    .select(
        "transaction_id",
        "transaction_date",
        "transaction_year",
        "transaction_month",
        "total_amount"
    ) \
    .show(5, truncate=False)

+-------------+------------------------+-----------+
|namespace    |tableName               |isTemporary|
+-------------+------------------------+-----------+
|retailmart_db|feedback_table          |false      |
|retailmart_db|transactions_partitioned|false      |
|retailmart_db|transactions_table      |false      |
+-------------+------------------------+-----------+

Partitioned transaction table schema:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- store_location: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- loyalty_years: integer (nullable = tru

Step 12: Query Partitioned Data

In [19]:
# Query transactions for January 2026
jan_2026 = spark.sql("""
    SELECT transaction_id, transaction_date,
           product_category, total_amount
    FROM retailmart_db.transactions_partitioned
    WHERE transaction_year = 2026
      AND transaction_month = 1
""")

# Display sample results and count matching transactions
jan_2026.show(5, truncate=False)

print("Transactions in January 2026:", jan_2026.count())

+--------------+-------------------+----------------+------------+
|transaction_id|transaction_date   |product_category|total_amount|
+--------------+-------------------+----------------+------------+
|TXN-00000013  |2026-01-13 17:25:23|Beauty          |26.32       |
|TXN-00000028  |2026-01-15 14:30:00|Groceries       |14.02       |
|TXN-00000054  |2026-01-15 00:00:00|Sports          |105.46      |
|TXN-00000093  |2026-01-15 14:30:00|Books           |78.16       |
|TXN-00000153  |2026-01-15 14:30:00|Toys            |24.67       |
+--------------+-------------------+----------------+------------+
only showing top 5 rows

Transactions in January 2026: 29349


Step 13: Compare Storage Formats

In [20]:
# Define output directories
csv_path = "/content/retailmart_csv"
parquet_path = "/content/retailmart_parquet"
orc_path = "/content/retailmart_orc"

# Write the transaction dataset in each format
df_transactions.write.mode("overwrite").option("header", True).csv(csv_path)

df_transactions.write.mode("overwrite").parquet(parquet_path)

df_transactions.write.mode("overwrite").orc(orc_path)

print("CSV, Parquet, and ORC files written successfully!")

CSV, Parquet, and ORC files written successfully!


Step 14: Measure Storage Sizes

In [21]:
# Calculate the total size of each format in bytes
def get_directory_size(path):
    total_size = 0

    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)

    return total_size

# Calculate sizes in megabytes
storage_sizes = {
    "CSV": get_directory_size(csv_path) / (1024 * 1024),
    "Parquet": get_directory_size(parquet_path) / (1024 * 1024),
    "ORC": get_directory_size(orc_path) / (1024 * 1024)
}

# Display comparison
for format_name, size_mb in storage_sizes.items():
    print(f"{format_name}: {size_mb:.2f} MB")

CSV: 142.42 MB
Parquet: 57.55 MB
ORC: 56.56 MB


Step 15: Storage Efficiency Analysis

In [23]:
import pandas as pd

In [24]:
# Create a storage comparison table
csv_size = storage_sizes["CSV"]
parquet_size = storage_sizes["Parquet"]
orc_size = storage_sizes["ORC"]

storage_comparison = pd.DataFrame({
    "Format": ["CSV", "Parquet", "ORC"],
    "Size (MB)": [csv_size, parquet_size, orc_size],
    "Space Saved vs CSV (%)": [
        0,
        ((csv_size - parquet_size) / csv_size) * 100,
        ((csv_size - orc_size) / csv_size) * 100
    ]
})

print("Storage Format Comparison:")
display(storage_comparison.round(2))

Storage Format Comparison:


,Format,Size (MB),Space Saved vs CSV (%)
0,CSV,142.42,0.00
1,Parquet,57.55,59.60
2,ORC,56.56,60.29


Step 16: Query Performance Comparison

In [25]:
import time

# Query the regular transaction table
start_time = time.time()

regular_result = spark.sql("""
    SELECT COUNT(*) AS total
    FROM retailmart_db.transactions_table
    WHERE YEAR(transaction_date) = 2026
      AND MONTH(transaction_date) = 1
""").collect()

regular_time = time.time() - start_time

# Query the partitioned transaction table
start_time = time.time()

partitioned_result = spark.sql("""
    SELECT COUNT(*) AS total
    FROM retailmart_db.transactions_partitioned
    WHERE transaction_year = 2026
      AND transaction_month = 1
""").collect()

partitioned_time = time.time() - start_time

# Display results
print("Regular table count:", regular_result[0]["total"])
print(f"Regular table query time: {regular_time:.3f} seconds")

print("\nPartitioned table count:", partitioned_result[0]["total"])
print(f"Partitioned table query time: {partitioned_time:.3f} seconds")

Regular table count: 29349
Regular table query time: 3.894 seconds

Partitioned table count: 29349
Partitioned table query time: 0.500 seconds


Step 17: Architecture and Storage Findings

In [26]:
# Summarize storage and query performance results
final_results = pd.DataFrame({
    "Metric": [
        "CSV storage size (MB)",
        "Parquet storage size (MB)",
        "ORC storage size (MB)",
        "Regular query time (seconds)",
        "Partitioned query time (seconds)",
        "January 2026 transaction count"
    ],
    "Result": [
        round(csv_size, 2),
        round(parquet_size, 2),
        round(orc_size, 2),
        round(regular_time, 3),
        round(partitioned_time, 3),
        partitioned_result[0]["total"]
    ]
})

display(final_results)

,Metric,Result
0,CSV storage size (MB),142.420
1,Parquet storage size (MB),57.550
2,ORC storage size (MB),56.560
3,Regular query time (seconds),3.894
4,Partitioned query time (seconds),0.500
5,January 2026 transaction count,29349.000


Step 19: Final Validation

In [27]:
# Final validation of datasets and stored tables

print("TRANSACTION DATA")
print("Rows:", df_transactions.count())
print("Columns:", len(df_transactions.columns))

print("\nCUSTOMER FEEDBACK DATA")
print("Rows:", df_feedback.count())
print("Columns:", len(df_feedback.columns))

print("\nSTORED TABLES")
spark.sql("SHOW TABLES IN retailmart_db").show(truncate=False)

print("\nPARTITIONED TABLE COUNT")
print(
    "Rows:",
    spark.table("retailmart_db.transactions_partitioned").count()
)

print("\nAll final validation checks completed.")

TRANSACTION DATA
Rows: 973266
Columns: 15

CUSTOMER FEEDBACK DATA
Rows: 250000
Columns: 14

STORED TABLES
+-------------+------------------------+-----------+
|namespace    |tableName               |isTemporary|
+-------------+------------------------+-----------+
|retailmart_db|feedback_table          |false      |
|retailmart_db|transactions_partitioned|false      |
|retailmart_db|transactions_table      |false      |
+-------------+------------------------+-----------+


PARTITIONED TABLE COUNT
Rows: 973266

All final validation checks completed.
